In [7]:

import numpy as np                
import pandas as pd                
import matplotlib.pyplot as plt    
from pathlib import Path


from iminuit import Minuit

# Configurazione grafica 
plt.rcParams.update({
    'font.family': 'serif',        
    'mathtext.fontset': 'cm',      
    'figure.figsize': (10, 6),      
    'axes.grid': True,             
    'grid.linestyle': '--',        
    'grid.alpha': 0.6,             
    'errorbar.capsize': 3,         
    'lines.linewidth': 1.5,        
    'lines.markersize': 5          
})

In [ ]:
# CELLA 2: LETTURA DATI E CALCOLO INCERTEZZE
# Versione semplice e leggibile: cerca i dati numerici, poi usa le colonne x e V

nome_file = 'EsempioDati.csv'


def trova_colonna(df, aliases_x, aliases_y):
    cols = [str(c).strip().lower() for c in df.columns]
    x_col = None
    y_col = None

    for c in df.columns:
        name = str(c).strip().lower()
        if x_col is None and name in aliases_x:
            x_col = c
        if y_col is None and name in aliases_y:
            y_col = c

    if x_col is None and len(df.columns) >= 2:
        x_col = df.columns[0]
    if y_col is None and len(df.columns) >= 2:
        y_col = df.columns[1]

    return x_col, y_col


try:
    df = None
    for sep, dec in [(';', ','), (',', '.'), ('\t', '.')]:
        for header in [0, None]:
            try:
                tmp = pd.read_csv(nome_file, sep=sep, decimal=dec, header=header, engine='python')
                tmp = tmp.dropna(how='all').reset_index(drop=True)
                tmp = tmp.apply(pd.to_numeric, errors='coerce')
                tmp = tmp.dropna(how='all').reset_index(drop=True)
                if tmp.shape[1] >= 2 and not tmp.empty:
                    df = tmp
                    break
            except Exception:
                pass
        if df is not None:
            break

    if df is None:
        raise ValueError('Nessun CSV valido trovato.')

    x_aliases = {'x', 'raggio', 'distance', 'dist', 'posizione', 'pos'}
    y_aliases = {'v', 'voltage', 'irradianza', 'intensity', 'signal', 'y'}
    x_col, y_col = trova_colonna(df, x_aliases, y_aliases)

    if x_col is None or y_col is None:
        raise KeyError(f'Non riesco a capire quali colonne siano x e V. Colonne trovate: {list(df.columns)}')

    df2 = df[[x_col, y_col]].copy()
    df2.columns = ['x', 'V']
    df2 = df2.dropna(subset=['x', 'V']).reset_index(drop=True)

    x = df2['x'].to_numpy(dtype=float)
    V = df2['V'].to_numpy(dtype=float)

    err_x = np.full_like(x, 0.2 / np.sqrt(12), dtype=float)
    err_lettura = 0.01
    err_fondo = 0.005
    err_V = np.full_like(V, np.sqrt(err_lettura**2 + err_fondo**2), dtype=float)

    print(f'Letti correttamente {len(x)} punti validi.')
    print(df2.head())

except FileNotFoundError:
    print(f'File {nome_file} non trovato. Uso dati fittizi per testare il codice.')
    x = np.linspace(40, 500, 30)
    err_x = np.full_like(x, 0.2 / np.sqrt(12))
    err_V = np.full_like(x, 0.05)
    V = 5000 / (x - 12)**2 + 0.1 + np.random.normal(0, err_V, len(x))
    print('Dati fittizi generati correttamente.')


KeyError: 'x'

In [ ]:
# FIT A 3 PARAMETRI (A, B, C)

# Definizione del modello: f(x) = A / (x - B)^2 + C
def f3(x, A, B, C):
    return A / (x - B)**2 + C

# Derivata analitica di f(x) rispetto ad x (serve per la varianza efficace)
def df3(x, A, B, C):
    return -2 * A / (x - B)**3

# Funzione da minimizzare: Chi Quadrato con varianza efficace
def chi2_eff_3(A, B, C):
    # Propagazione dell'errore di x sulle ordinate
    sigma_eff = np.sqrt(err_V**2 + (err_x * df3(x, A, B, C))**2)
    return np.sum(((V - f3(x, A, B, C)) / sigma_eff)**2)

print("\n--- ESECUZIONE FIT 3 PARAMETRI ---")
# Parametri iniziali (guess). Aggiustali se il fit non converge!
m3 = Minuit(chi2_eff_3, A=5000, B=10, C=0.1)
m3.errordef = Minuit.LEAST_SQUARES
m3.migrad()
m3.hesse()

# Estrazione Risultati
A_fit, B_fit, C_fit = m3.values["A"], m3.values["B"], m3.values["C"]
err_A, err_B, err_C = m3.errors["A"], m3.errors["B"], m3.errors["C"]
chi2_3 = m3.fval
ndof_3 = len(x) - 3

print(m3)

# PLOT DEL FIT
plt.figure()
plt.errorbar(x, V, xerr=err_x, yerr=err_V, fmt='ko', label='Dati sperimentali', zorder=1)

x_plot = np.linspace(min(x)-10, max(x)+10, 500)
plt.plot(x_plot, f3(x_plot, A_fit, B_fit, C_fit), 'r-', lw=2, label='Fit: $A/(x-B)^2 + C$', zorder=2)

# Asintoti B e C
plt.axvline(B_fit, color='blue', linestyle='-.', alpha=0.5, label=f'Asintoto B = {B_fit:.2f} mm')
plt.axhline(C_fit, color='green', linestyle='-.', alpha=0.5, label=f'Fondo C = {C_fit:.2f} V')

# Box di testo coi risultati
res_testo = (f"$\chi^2/ndof$ = {chi2_3:.2f} / {ndof_3}\n"
             f"$A$ = {A_fit:.1f} $\pm$ {err_A:.1f}\n"
             f"$B$ = {B_fit:.2f} $\pm$ {err_B:.2f} mm\n"
             f"$C$ = {C_fit:.3f} $\pm$ {err_C:.3f} V")
plt.text(0.95, 0.95, res_testo, transform=plt.gca().transAxes, 
         fontsize=12, verticalalignment='top', horizontalalignment='right', 
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.xlabel('Posizione x [mm]', fontsize=14)
plt.ylabel('Tensione (Segnale Netto) V [Volt]', fontsize=14)
plt.title('Fit dell\'irradianza in funzione della distanza', fontsize=16)
plt.legend()
plt.show()

Valori validi: x=22, y=19, z=5
[ 1.  1.  1.  1.  1.  1. 11.  1.  1.  1.  1.  1.  1.  1. 11.  1.  1.  1.
  1. 11.  1.  1.]
[1.00000e+00 5.50000e+00 6.40000e+01 4.30000e+01 4.22300e+01 3.15000e+02
 5.46000e+02 4.00000e+00 7.57000e+02 4.00000e+00 3.60000e+01 3.60000e+01
 4.30000e+01 4.50000e+01 0.00000e+00 3.20000e+01 5.30000e+01 3.52352e+05
 6.40000e+01]
[32. 45. 45.  5.  5.]


In [ ]:
# CELLA 4: ANALISI DEI RESIDUI

# Calcolo residui e varianza efficace punto per punto
residui = V - f3(x, A_fit, B_fit, C_fit)
sigma_eff = np.sqrt(err_V**2 + (err_x * df3(x, A_fit, B_fit, C_fit))**2)
residui_norm = residui / sigma_eff

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

#  GRAFICO 1: ISTOGRAMMA E FIT GAUSSIANO 
counts, bins, _ = ax1.hist(residui_norm, bins=8, range=(-3, 3), alpha=0.6, color='steelblue', edgecolor='black')

def gauss(x, norm, mu, sigma):
    return norm * np.exp(-0.5 * ((x - mu) / sigma)**2)

def chi2_gauss(norm, mu, sigma):
    mask = counts > 0
    x_mid = (bins[:-1] + bins[1:])[mask] / 2
    y_teo = gauss(x_mid, norm, mu, sigma)
    err_y = np.sqrt(counts[mask])
    return np.sum(((counts[mask] - y_teo) / err_y)**2)

mg = Minuit(chi2_gauss, norm=max(counts), mu=0.0, sigma=1.0)
mg.errordef = Minuit.LEAST_SQUARES
mg.migrad()

x_g = np.linspace(-4, 4, 100)
ax1.plot(x_g, gauss(x_g, mg.values['norm'], mg.values['mu'], mg.values['sigma']), 'r-', lw=2, 
         label=f"$\mu$ = {mg.values['mu']:.2f} $\pm$ {mg.errors['mu']:.2f}\n$\sigma$ = {mg.values['sigma']:.2f} $\pm$ {mg.errors['sigma']:.2f}")
ax1.set_title("Istogramma dei Residui Normalizzati")
ax1.set_xlabel("Residuo / $\sigma_{eff}$")
ax1.set_ylabel("Conteggi")
ax1.legend()

#  GRAFICO 2: RESIDUI VS X (Ricerca trend) 
ax2.errorbar(x, residui, yerr=sigma_eff, fmt='o', color='black', alpha=0.7)
ax2.axhline(0, color='red', linestyle='--', lw=2)

# Fit lineare (per verificare P1 ~ 0)
def retta(x, P0, P1): return P0 + P1 * x
def chi2_retta(P0, P1): return np.sum(((residui - retta(x, P0, P1)) / sigma_eff)**2)

ml = Minuit(chi2_retta, P0=0.0, P1=0.0)
ml.errordef = Minuit.LEAST_SQUARES
ml.migrad()

ax2.plot(x, retta(x, ml.values['P0'], ml.values['P1']), 'b-', alpha=0.5, 
         label=f"Pendenza = {ml.values['P1']:.1e} $\pm$ {ml.errors['P1']:.1e}")
ax2.set_title("Residui vs Posizione x")
ax2.set_xlabel("Posizione x [mm]")
ax2.set_ylabel("Residuo [V]")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# CELLA 5: FIT A 4 PARAMETRI (Esponente Libero)

# Definizione del modello: f(x) = A / (x - B)^D + C
def f4(x, A, B, C, D):
    return A / (x - B)**D + C

def df4(x, A, B, C, D):
    return -D * A / (x - B)**(D + 1)

def chi2_eff_4(A, B, C, D):
    sigma_eff = np.sqrt(err_V**2 + (err_x * df4(x, A, B, C, D))**2)
    return np.sum(((V - f4(x, A, B, C, D)) / sigma_eff)**2)

print("\n--- ESECUZIONE FIT CON ESPONENTE LIBERO ---")
# Usiamo i valori ottimali trovati prima come guess iniziali
m4 = Minuit(chi2_eff_4, A=A_fit, B=B_fit, C=C_fit, D=2.0)
m4.errordef = Minuit.LEAST_SQUARES
m4.migrad()
m4.hesse()

D_fit = m4.values["D"]
err_D = m4.errors["D"]

print(m4)
print(f"\n---> RISULTATO FISICO: L'esponente misurato è D = {D_fit:.3f} +/- {err_D:.3f}")

In [ ]:
# CELLA 6: VERIFICA COMPATIBILITA' PARAMETRO B

print("--- VERIFICA PARAMETRO B CON ERRORI SISTEMATICI ---")

# Parametro B dal primo fit (3 parametri)
B_misurato = B_fit
err_B_stat = err_B

# Errori sistematici stimati (MODIFICA)
err_x0 = 0.5      # metà tacca al contatto
err_D_mezzi = 0.5 # errore misurazione raggio lampadina col calibro
err_prof = 0.5    # stima errore profondità sensore (1 mm +/- 0.5)

# Somma in quadratura dell'errore totale su B
err_B_sist = np.sqrt(err_x0**2 + err_D_mezzi**2 + err_prof**2)
err_B_tot = np.sqrt(err_B_stat**2 + err_B_sist**2)

print(f"Valore dal Fit: B = {B_misurato:.2f} mm")
print(f"Errore Statistico (dal fit)   = {err_B_stat:.2f} mm")
print(f"Errore Sistematico stimato    = {err_B_sist:.2f} mm")
print(f"--> ERRORE TOTALE SU B        = {err_B_tot:.2f} mm")

# Test di compatibilità con 0 (quante sigma di distanza?)
t_student = abs(B_misurato - 0) / err_B_tot
print(f"\nDistanza dallo zero: {t_student:.1f} sigma")
if t_student < 3:
    print("CONCLUSIONE: Il parametro B è fisicamente compatibile con le tolleranze dell'apparato.")
else:
    print("CONCLUSIONE: B non è compatibile con zero. Potrebbe esserci un errore di allineamento h != 0.")